## GAWP Sandbox

Experiments with reading data files for germline analysis

In [1]:
import os
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_DIR = Path.home() / 'Research/Projects/LibudaLab/GAP/PRG-1'
DATA_DIR = PROJECT_DIR / 'Imaris'
OUTPUT_DIR = PROJECT_DIR / 'Demo'
MEASUREMENTS = PROJECT_DIR / 'Germline Measurements.xlsx'


In [3]:
DATA_DIR

PosixPath('/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Imaris')

#### Data Format

The data directory has three spreadsheets:

```shell
xlsx
├── 210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx
├── 210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif.xlsx
└── 210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched.xlsx
```

When viewed in Excel there are 39 tabs:
* Overall describes the data
* Intensity max, mean, and sum for three channels (9 tabs in all)
* Position X, Y, and Z and combined
* Sphericity, Volume, distances



### Read Data File

Get a list of file names in the data directory.

In [4]:
data_files = sorted(os.listdir(DATA_DIR))

In [5]:
data_files

['210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx',
 '210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitched.xlsx',
 '210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched.xlsx']

This expression is the path to the first data file

In [6]:
DATA_DIR / data_files[0]

PosixPath('/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Imaris/210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx')

#### Experiment #1

With no other arguments we expect the Pandas method that reads XLS files to return the first sheet in the spreadsheet.

```python
> df = pd.read_excel(DATA_DIR / data_files[0])
```

That fails with `ValueError: Worksheet index 0 is invalid, 0 worksheets found`

#### Experiment #2

It looks like we have older "macro-enabled" (`.xlsm`) files.  The suggested fix is to use a different parsing engine.

In [7]:
df = pd.read_excel(DATA_DIR / data_files[0], engine='calamine')

In [8]:
type(df)

pandas.DataFrame

That worked.  We got the first ("Overall") sheet that lists the measurements included in the file.

There is an issue, though -- the first row has the sheet name, and the column names are in the second row.  Pass another paramter to tell Pandas to get the column names from the second row.

In [9]:
df = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1)

In [10]:
df[0:4]

,Variable,Value,Unit,Channel,Image,Name,Surpass Object,Time,ID
0,Data Intensity Max,46125.000000,NaN,1.0,Image 1,NaN,Volume,1.0,NaN
1,Data Intensity Max,42515.000000,NaN,2.0,Image 1,NaN,Volume,1.0,NaN
2,Data Intensity Max,15856.000000,NaN,3.0,Image 1,NaN,Volume,1.0,NaN
3,Data Intensity Mean,816.711975,NaN,1.0,Image 1,NaN,Volume,1.0,NaN


Get the frame with max intensity data (channel 1)

In [11]:
imax_1 = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1, sheet_name='Intensity Max Ch=1 Img=1')

In [12]:
len(imax_1)

4579

In [13]:
imax_1.head()

,Intensity Max,Unit,Category,Channel,Image,Surpass Object,Time,ID
0,6754,NaN,Surface,1,Image 1,prg1_tz,1,0
1,4964,NaN,Surface,1,Image 1,prg1_ep,1,0
2,6293,NaN,Surface,1,Image 1,prg1_mp,1,0
3,9391,NaN,Surface,1,Image 1,prg1_lp,1,0
4,6072,NaN,Surface,1,Image 1,prg1_dk,1,0


#### `ExcelFile` Object

Since we're reading multiple sheets we want to use the `ExcelFile` class.  It opens the file and loads it into memory, and then we can call a file method to parse each sheet.

This statement opens the file and saves it so we can examine its properties.  One useful property is a list of sheet names.

In [14]:
f = pd.ExcelFile(DATA_DIR / data_files[0], engine='calamine')

In [15]:
type(f)

pandas.ExcelFile

In [16]:
len(f.sheet_names)

39

In [17]:
f.sheet_names[0]

'Overall'

In [18]:
dir(f)

['CalamineReader',
 'ODFReader',
 'OpenpyxlReader',
 'PyxlsbReader',
 'XlrdReader',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__fspath__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_engines',
 '_io',
 '_module_source',
 '_reader',
 'book',
 'close',
 'engine',
 'parse',
 'sheet_names',
 'storage_options']

In [19]:
f.parse('Position')

,Position,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
1,21.794001,138.720001,7.7,µm,Surface,Position,NaN,prg1_dk,1,0
2,19.615,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
3,26.146,135.149994,7.9,µm,Surface,Position,NaN,prg1_dk,1,2
4,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
...,...,...,...,...,...,...,...,...,...,...
4592,45.424,19.481001,2.1,µm,MeasurementPoint,Position,M,Measurement Points 1,1,12
4593,41.882999,26.756001,2.7,µm,MeasurementPoint,Position,N,Measurement Points 1,1,13
4594,36.717999,37.508999,3.5,µm,MeasurementPoint,Position,O,Measurement Points 1,1,14
4595,23.646,69.875,6.3,µm,MeasurementPoint,Position,P,Measurement Points 1,1,15


### Output File

In [20]:
OUTPUT_DIR

PosixPath('/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Demo')

In [21]:
! tree /Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Demo

/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Demo
├── Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched_local.xlsx
├── Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif_local.xlsx
└── Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched_local.xlsx

1 directory, 3 files


So we have one output for each input, where the names has "Linearized" prepended to the input file name.

In [22]:
output_files = sorted(os.listdir(OUTPUT_DIR))

In [23]:
output_files

['Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched_local.xlsx',
 'Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif_local.xlsx',
 'Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched_local.xlsx']

In [24]:
ddf = pd.read_excel(OUTPUT_DIR/output_files[0], engine='calamine')

In [25]:
len(ddf)

4583

In [26]:
ddf.head()

,Unit,Category,Collection,Name,Surpass Object,Time,ID,GonadID,X,Y,...,displacement,distance1,distance2,delta_distance1,delta_distance2,segment,tX,ntX,meiosis_stage,ntX_pachytene_normd
0,µm,prg1_dk,Position,NaN,prg1_dk,1,0,NaN,21.794001,138.720001,...,16.818130,2.058651,18.945127,-0.901987,-0.098013,1,2.058651,0.008604,PMT,0.008604
1,µm,prg1_dk,Position,NaN,prg1_dk,1,2,NaN,26.146000,135.149994,...,12.633132,5.823043,15.180735,-0.722762,-0.277238,1,5.823043,0.024337,PMT,0.024337
2,µm,prg1_dk,Position,NaN,prg1_dk,1,4,NaN,25.013000,137.296005,...,13.667280,3.627686,17.376091,-0.827284,-0.172716,1,3.627686,0.015162,PMT,0.015162
3,µm,prg1_dk,Position,NaN,prg1_dk,1,6,NaN,24.264999,138.662994,...,14.352286,2.228068,18.775710,-0.893921,-0.106079,1,2.228068,0.009312,PMT,0.009312
4,µm,prg1_dk,Position,NaN,prg1_dk,1,7,NaN,26.162001,137.666000,...,12.502630,3.310372,17.693406,-0.842392,-0.157608,1,3.310372,0.013836,PMT,0.013836


### Measurements

The coordinates of the points that mark the center of a germline are in the "Position" sheet.

In [27]:
data_files[0]

'210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx'

In [28]:
mf = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1, sheet_name="Position")

In [29]:
mf.head()

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
0,21.794001,138.720001,7.700,µm,Surface,Position,NaN,prg1_dk,1,0
1,19.615000,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
2,26.146000,135.149994,7.900,µm,Surface,Position,NaN,prg1_dk,1,2
3,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
4,25.013000,137.296005,7.400,µm,Surface,Position,NaN,prg1_dk,1,4


The Category column has "Surface" for a data point or "MeasurementPoint" for a end-point of a line segment.

In [30]:
sf = mf[mf.Category == "MeasurementPoint"]

In [31]:
sf.head()

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
4579,38.500999,141.542007,0.521,µm,MeasurementPoint,Position,A,Measurement Points 1,1,0
4580,39.457001,120.559998,6.300,µm,MeasurementPoint,Position,B,Measurement Points 1,1,1
4581,40.560001,110.825996,4.700,µm,MeasurementPoint,Position,C,Measurement Points 1,1,2
4582,43.907001,99.404999,4.300,µm,MeasurementPoint,Position,D,Measurement Points 1,1,3
4583,48.821999,81.403999,4.900,µm,MeasurementPoint,Position,E,Measurement Points 1,1,4


In [32]:
sf[['Name','Position X','Position Y']]

,Name,Position X,Position Y
4579,A,38.500999,141.542007
4580,B,39.457001,120.559998
4581,C,40.560001,110.825996
4582,D,43.907001,99.404999
4583,E,48.821999,81.403999
4584,F,54.426998,66.403999
4585,G,61.472000,53.318001
4586,H,67.041000,44.591000
4587,I,72.586998,25.798000
4588,J,69.010002,14.931000


Save the measurement names and locations in a CSV file

In [33]:
sf[['Position X','Position Y','Name']].to_csv("measurements.csv", index=False)

Save a random sample of point locations to use in tests

In [34]:
mf[mf.Category=='Surface'].sample(20).to_csv("points.csv", index=False)

### Nucleus IDs

D'oh, first version overlooked the fact that the "ID" column is not a unique ID.  We need to concatenate the "Surpass Object" column (strings of the form `prg1_dk`, `prg1_ep`, ...) with the number in the ID column.

In [35]:
pf = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1, sheet_name='Position')

In [36]:
pf

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
0,21.794001,138.720001,7.700,µm,Surface,Position,NaN,prg1_dk,1,0
1,19.615000,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
2,26.146000,135.149994,7.900,µm,Surface,Position,NaN,prg1_dk,1,2
3,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
4,25.013000,137.296005,7.400,µm,Surface,Position,NaN,prg1_dk,1,4
...,...,...,...,...,...,...,...,...,...,...
4591,45.424000,19.481001,2.100,µm,MeasurementPoint,Position,M,Measurement Points 1,1,12
4592,41.882999,26.756001,2.700,µm,MeasurementPoint,Position,N,Measurement Points 1,1,13
4593,36.717999,37.508999,3.500,µm,MeasurementPoint,Position,O,Measurement Points 1,1,14
4594,23.646000,69.875000,6.300,µm,MeasurementPoint,Position,P,Measurement Points 1,1,15


In [37]:
pf['Surpass Object'] + pf['ID'].apply(str)

0                     prg1_dk0
1                     prg1_dk1
2                     prg1_dk2
3                     prg1_dk3
4                     prg1_dk4
                 ...          
4591    Measurement Points 112
4592    Measurement Points 113
4593    Measurement Points 114
4594    Measurement Points 115
4595    Measurement Points 116
Length: 4596, dtype: str

In [38]:
pd.Series(pf['Surpass Object'] + pf['ID'].apply(str), name='ID')

0                     prg1_dk0
1                     prg1_dk1
2                     prg1_dk2
3                     prg1_dk3
4                     prg1_dk4
                 ...          
4591    Measurement Points 112
4592    Measurement Points 113
4593    Measurement Points 114
4594    Measurement Points 115
4595    Measurement Points 116
Name: ID, Length: 4596, dtype: str

In [39]:
pf

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
0,21.794001,138.720001,7.700,µm,Surface,Position,NaN,prg1_dk,1,0
1,19.615000,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
2,26.146000,135.149994,7.900,µm,Surface,Position,NaN,prg1_dk,1,2
3,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
4,25.013000,137.296005,7.400,µm,Surface,Position,NaN,prg1_dk,1,4
...,...,...,...,...,...,...,...,...,...,...
4591,45.424000,19.481001,2.100,µm,MeasurementPoint,Position,M,Measurement Points 1,1,12
4592,41.882999,26.756001,2.700,µm,MeasurementPoint,Position,N,Measurement Points 1,1,13
4593,36.717999,37.508999,3.500,µm,MeasurementPoint,Position,O,Measurement Points 1,1,14
4594,23.646000,69.875000,6.300,µm,MeasurementPoint,Position,P,Measurement Points 1,1,15


In [40]:
id_cols = pf[['Surpass Object','ID','Name']]
id_cols

,Surpass Object,ID,Name
0,prg1_dk,0,NaN
1,prg1_dk,1,NaN
2,prg1_dk,2,NaN
3,prg1_dk,3,NaN
4,prg1_dk,4,NaN
...,...,...,...
4591,Measurement Points 1,12,M
4592,Measurement Points 1,13,N
4593,Measurement Points 1,14,O
4594,Measurement Points 1,15,P


This fails because not all values are strings:

In [41]:
# id_cols.agg("sum", axis=1)

This seems clunky but it works.  Initialize a series with strings made from the first column, then iterate over the remaining columns, converting the column to strings and appending it to the series.

In [42]:
col_names = ['Surpass Object','ID','Name']

In [43]:
ids = id_cols[col_names[0]].apply(str)
ids

0                    prg1_dk
1                    prg1_dk
2                    prg1_dk
3                    prg1_dk
4                    prg1_dk
                ...         
4591    Measurement Points 1
4592    Measurement Points 1
4593    Measurement Points 1
4594    Measurement Points 1
4595    Measurement Points 1
Name: Surpass Object, Length: 4596, dtype: str

In [44]:
for col in col_names[1:]:
    s = id_cols[col].apply(str)
    ids += '.' + s

In [45]:
ids

0                   prg1_dk.0.nan
1                   prg1_dk.1.nan
2                   prg1_dk.2.nan
3                   prg1_dk.3.nan
4                   prg1_dk.4.nan
                  ...            
4591    Measurement Points 1.12.M
4592    Measurement Points 1.13.N
4593    Measurement Points 1.14.O
4594    Measurement Points 1.15.P
4595    Measurement Points 1.16.Q
Name: Surpass Object, Length: 4596, dtype: str

In [46]:
list('foo')

['f', 'o', 'o']

In [47]:
list((0,1))

[0, 1]

In [48]:
pf

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
0,21.794001,138.720001,7.700,µm,Surface,Position,NaN,prg1_dk,1,0
1,19.615000,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
2,26.146000,135.149994,7.900,µm,Surface,Position,NaN,prg1_dk,1,2
3,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
4,25.013000,137.296005,7.400,µm,Surface,Position,NaN,prg1_dk,1,4
...,...,...,...,...,...,...,...,...,...,...
4591,45.424000,19.481001,2.100,µm,MeasurementPoint,Position,M,Measurement Points 1,1,12
4592,41.882999,26.756001,2.700,µm,MeasurementPoint,Position,N,Measurement Points 1,1,13
4593,36.717999,37.508999,3.500,µm,MeasurementPoint,Position,O,Measurement Points 1,1,14
4594,23.646000,69.875000,6.300,µm,MeasurementPoint,Position,P,Measurement Points 1,1,15


In [50]:
type(pf)

pandas.DataFrame